# BlobNet quickstart: simulate, load models, and run inference

This notebook loads the publication checkpoints for Square-Net, Hex-Net, and Blob-Net, evaluates all three on one deterministic simulated image, and ends with a block for trying an experimental image or your own data. Run it from the repository environment with `uv run jupyter lab`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyTEMlib.file_tools
import torch
import yaml
from scipy.ndimage import gaussian_filter

from blobnet.metrics import evaluate_heatmap_localization, extract_subpixel_peak_positions
from blobnet.networks import build_unet
from blobnet.synthetic import RandomAtomImageConfig, generate_atom_image

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').is_file(), 'Run this notebook from the BlobNet repository or notebooks/ directory.'
ROOT

## Load the three publication models

The checkpoints are tracked under `artifacts/manuscript_models`. The models share the same U-Net architecture and differ only in their training geometry.

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

def load_model(checkpoint_path):
    model = build_unet(num_filters=[32, 64, 128, 256], dropout=0.2)
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    state = checkpoint.get('model_state_dict', checkpoint)
    model.load_state_dict(state)
    return model.to(device).eval()

checkpoint_root = ROOT / 'artifacts' / 'manuscript_models'
models = {
    'Square-Net': load_model(checkpoint_root / 'square' / 'unet_best.pth'),
    'Hex-Net': load_model(checkpoint_root / 'hexagonal' / 'unet_best.pth'),
    'Blob-Net': load_model(checkpoint_root / 'random' / 'unet_best.pth'),
}
print(f'Loaded {len(models)} models on {device}.')

In [ ]:
def normalize_image(image, low=1.0, high=99.8):
    image = np.asarray(image, dtype=np.float32)
    lo, hi = np.percentile(image[np.isfinite(image)], [low, high])
    return np.clip((image - lo) / max(hi - lo, 1e-8), 0, 1).astype(np.float32)

def predict(model, image):
    tensor = torch.from_numpy(np.asarray(image, dtype=np.float32))[None, None].to(device)
    with torch.inference_mode():
        return torch.sigmoid(model(tensor))[0, 0].cpu().numpy()

def predict_all(image):
    return {name: predict(model, image) for name, model in models.items()}

## Generate one simulated image

This uses the tracked random-dataset configuration and a fixed seed, so everyone receives the same image and ground-truth atom coordinates.

In [ ]:
config_data = yaml.safe_load((ROOT / 'configs/dataset_configs/random.yaml').read_text())
parameters = dict(config_data['parameters'])
# A 256 px example runs quickly while retaining the publication image model.
parameters.update(image_shape=(256, 256), min_atoms=260, max_atoms=320)
config = RandomAtomImageConfig(**parameters)
sample = generate_atom_image(config, np.random.default_rng(2026))
simulated_image = sample['image']
ground_truth = sample['coordinates']
print(f'Simulated image: {simulated_image.shape}; ground-truth atoms: {len(ground_truth)}')

In [ ]:
simulated_predictions = predict_all(simulated_image)
simulated_results = {
    name: evaluate_heatmap_localization(
        heatmap, ground_truth, threshold_rel=0.35, min_distance=3, match_distance=3.0
    )
    for name, heatmap in simulated_predictions.items()
}
for name, result in simulated_results.items():
    print(f"{name:10s} F1={result['f1']:.3f}  precision={result['precision']:.3f}  "
          f"recall={result['recall']:.3f}  RMSE={result['rmse']:.3f} px")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
axes[0].imshow(simulated_image, cmap='gray')
axes[0].scatter(ground_truth[:, 1], ground_truth[:, 0], s=8, facecolors='none', edgecolors='#00E5FF', linewidths=0.5)
axes[0].set_title('Simulated input + truth')
for axis, (name, heatmap) in zip(axes[1:], simulated_predictions.items()):
    positions = simulated_results[name]['predicted_coordinates']
    axis.imshow(heatmap, cmap='magma', vmin=0, vmax=1)
    axis.scatter(positions[:, 1], positions[:, 0], s=8, facecolors='none', edgecolors='cyan', linewidths=0.5)
    axis.set_title(name)
for axis in axes:
    axis.set_axis_off()
plt.show()

## Try an experimental image or your own data

Edit only `path` below. The example is a pyTEMlib/NSID file included in this repository. `open_file` returns a dictionary of datasets; select the channel containing your image if your file has more than one. The example applies the same simple background suppression to every model.

In [ ]:
# Change this path to your own image.
path = str(ROOT / 'experimental_data/pristine_monolayer_MoS2.hf5')
# path = '/absolute/path/to/your/data.dm4'

datasets = pyTEMlib.file_tools.open_file(path)
raw = np.asarray(datasets['Channel_000'], dtype=np.float32).squeeze()
# Center-crop large images to 512 x 512 for this simple example.
crop_size = min(512, raw.shape[0], raw.shape[1])
crop_size -= crop_size % 8  # U-Net has three downsampling stages
if crop_size < 8:
    raise ValueError(f'Image is too small for inference: {raw.shape}')
y0 = (raw.shape[0] - crop_size) // 2
x0 = (raw.shape[1] - crop_size) // 2
raw = raw[y0:y0+crop_size, x0:x0+crop_size]
processed = normalize_image(gaussian_filter(normalize_image(raw), 1) - gaussian_filter(normalize_image(raw), 10))
experimental_predictions = predict_all(processed)
experimental_positions = {
    name: extract_subpixel_peak_positions(heatmap, threshold_rel=0.35, min_distance=3, window_size=5)
    for name, heatmap in experimental_predictions.items()
}
print(path, raw.shape, {name: len(points) for name, points in experimental_positions.items()})

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
axes[0].imshow(processed, cmap='gray')
axes[0].set_title('Processed input')
for axis, (name, heatmap) in zip(axes[1:], experimental_predictions.items()):
    points = experimental_positions[name]
    axis.imshow(processed, cmap='gray')
    axis.scatter(points[:, 1], points[:, 0], s=12, facecolors='none', edgecolors='#FF8C00', linewidths=0.7)
    axis.set_title(f'{name}: {len(points)} detections')
for axis in axes:
    axis.set_axis_off()
plt.show()